In [ ]:
#| default_exp handlers.pipeline.loader

# Loader

`HandlerConfig`, `load_data`, and `gap_check` — the first gate in the GeneralHandler pipeline.

In [ ]:
#| export
from __future__ import annotations
import io
import importlib
import importlib.util
from pathlib import Path
from typing import Annotated, Any, Optional, Union
import requests
import yaml
import pandas as pd
from pydantic import BaseModel, Field, ConfigDict, TypeAdapter, ValidationError


## HandlerConfig

Pydantic model that binds every YAML field with types; `from_yaml` is the single entry point.

In [ ]:
#| export
# Modules scanned (in order) when spec.name is used; first hit wins.
_SHARED_SCAN_MODULES = [
    "marisco.callbacks.shared",
    "marisco.callbacks.core",
]

_RECIPE_HINTS = {
    frozenset({"LAT", "LON"}): "config/recipes/recipe_split_lat_lon.yaml",
    frozenset({"VALUE", "UNC"}): "config/recipes/recipe_extract_value_unit.yaml",
}

def _yaml_step1_message(err: ValidationError) -> str:
    details = []
    for item in err.errors():
        loc = ".".join(str(p) for p in item["loc"])
        details.append(f"- {loc}: {item['msg']}")
    body = "\n".join(details) or "- YAML contract could not be validated."
    return (
        "Step 1: Type mismatch or missing fields detected in your YAML contract.\n"
        "Please fix the YAML configuration first; runtime callback guidance is withheld.\n"
        f"{body}"
    )

class _HandlerSection(BaseModel):
    module_name: str
    title: str = ""
    description: str = ""

class _DataSourceSection(BaseModel):
    url: str
    fname_out: str
    zenodo_id: str = ""
    format: str = "csv"

class _RenameColsSection(BaseModel):
    mapping: dict[str, str] = Field(default_factory=dict)
    string_cast: list[str] = Field(default_factory=list)

class _ParseDateTimeSection(BaseModel):
    col_date: Optional[str] = None
    col_time: Optional[str] = None
    format: str = "%Y-%m-%d"

class _MeltSection(BaseModel):
    meta_cols: list[str] = Field(default_factory=list)
    spec: list[dict[str, Any]] = Field(default_factory=list)

class _NomenclaturesSection(BaseModel):
    nuclide_lut: dict[str, int] = Field(default_factory=dict)
    unit_lut: dict[str, int] = Field(default_factory=dict)
    lab_lut: dict[str, int] = Field(default_factory=dict)
    lab_constants: dict[str, str] = Field(default_factory=dict)
    area_default: int = 0

class _OutputSection(BaseModel):
    keywords: list[str] = Field(default_factory=list)

class _RawHandlerContract(BaseModel):
    handler: _HandlerSection
    data_source: _DataSourceSection
    rename_cols: _RenameColsSection = Field(default_factory=_RenameColsSection)
    columns: dict[str, str] = Field(default_factory=dict)
    normalize_case: dict[str, str] = Field(default_factory=dict)
    parse_datetime: _ParseDateTimeSection = Field(default_factory=_ParseDateTimeSection)
    time_format: Optional[str] = None
    melt: _MeltSection = Field(default_factory=_MeltSection)
    unit_conversions: list[dict[str, Any]] = Field(default_factory=list)
    nomenclatures: _NomenclaturesSection = Field(default_factory=_NomenclaturesSection)
    output: _OutputSection = Field(default_factory=_OutputSection)
    loader: Optional[dict[str, Any]] = None
    pre_cbs: list[dict[str, Any]] = Field(default_factory=list)
    post_cbs: list[dict[str, Any]] = Field(default_factory=list)

class BasePluginSpec(BaseModel):
    "External Callback injection spec with polymorphic resolution."
    model_config = ConfigDict(populate_by_name=True)
    path: Optional[str] = None
    name: Optional[str] = None
    file: Optional[str] = None
    class_: Optional[str] = Field(None, alias="class")
    args: dict = Field(default_factory=dict)

    def resolve(self, yaml_dir: Path = None):
        raise NotImplementedError

    def resolve_fn(self, yaml_dir: Path = None):
        raise ValueError("Custom loader PluginSpec must use 'path:' (function, not class).")

class LegacyPathPluginSpec(BasePluginSpec):
    "Legacy fully-qualified dotted import path."
    path: str

    def resolve(self, yaml_dir: Path = None):
        module_path, class_name = self.path.rsplit(".", 1)
        return getattr(importlib.import_module(module_path), class_name)

    def resolve_fn(self, yaml_dir: Path = None):
        module_path, fn_name = self.path.rsplit(".", 1)
        return getattr(importlib.import_module(module_path), fn_name)

class NamePluginSpec(BasePluginSpec):
    "Shorthand callback name auto-resolved from shared/core modules."
    name: str

    def resolve(self, yaml_dir: Path = None):
        for mod_path in _SHARED_SCAN_MODULES:
            mod = importlib.import_module(mod_path)
            if hasattr(mod, self.name):
                return getattr(mod, self.name)
        raise ImportError(
            f"Callback '{self.name}' not found in {_SHARED_SCAN_MODULES}. "
            "Use the full 'path:' form to specify a non-shared callback."
        )

class FilePluginSpec(BasePluginSpec):
    "Local file import relative to the YAML directory."
    file: str
    class_: str = Field(alias="class")

    def resolve(self, yaml_dir: Path = None):
        if yaml_dir is None:
            raise ValueError("yaml_dir is required for file-based plugin loading")
        file_path = (Path(yaml_dir) / self.file).resolve()
        if not file_path.exists():
            raise FileNotFoundError(f"Plugin file not found: {file_path}")
        mod_spec = importlib.util.spec_from_file_location("_marisco_local_cb", file_path)
        mod = importlib.util.module_from_spec(mod_spec)
        mod_spec.loader.exec_module(mod)
        if not hasattr(mod, self.class_):
            raise AttributeError(f"Class '{self.class_}' not found in {file_path}")
        return getattr(mod, self.class_)

PluginSpecModel = Annotated[
    Union[LegacyPathPluginSpec, NamePluginSpec, FilePluginSpec],
    Field(union_mode="smart"),
]
_PLUGIN_SPEC_ADAPTER = TypeAdapter(PluginSpecModel)

class PluginSpec:
    "Backward-compatible factory for polymorphic plugin specs."

    def __new__(cls, *args, **kwargs):
        data = args[0] if args else kwargs
        return _PLUGIN_SPEC_ADAPTER.validate_python(data)

    @classmethod
    def model_validate(cls, data):
        return _PLUGIN_SPEC_ADAPTER.validate_python(data)

class MeltEntry(BaseModel):
    "One wide-to-long mapping entry: value column, uncertainty column, nuclide, unit, and lab."
    val:     str
    unc:     str
    nuclide: str
    unit:    str
    lab:     str

class UnitConversionCfg(BaseModel):
    "Single unit-conversion rule with its physical factor and optional metadata."
    nuclide:     str
    src_unit:    str
    dst_unit:    str
    factor:      float
    factor_name: str = ""
    comment:     str = ""

class HandlerConfig(BaseModel):
    "Complete handler configuration loaded from a YAML data-contract file."
    module_name: str
    title:       str = ""
    description: str = ""
    url:       str
    fname_out: str
    zenodo_id: str = ""
    fmt:       str = "csv"
    rename:      dict[str, str] = Field(default_factory=dict)
    string_cast: list[str]      = Field(default_factory=list)
    columns:     dict[str, str] = Field(default_factory=dict)
    normalize_case: dict[str, str] = Field(default_factory=dict)
    col_date:    Optional[str] = None
    col_time:    Optional[str] = None
    dt_format:   str           = "%Y-%m-%d"
    time_format: Optional[str] = None
    meta_cols: list[str]             = Field(default_factory=list)
    melt_spec: list[MeltEntry]       = Field(default_factory=list)
    unit_conversions: list[UnitConversionCfg] = Field(default_factory=list)
    nuclide_lut:   dict[str, int] = Field(default_factory=dict)
    unit_lut:      dict[str, int] = Field(default_factory=dict)
    lab_lut:       dict[str, int] = Field(default_factory=dict)
    lab_constants: dict[str, str] = Field(default_factory=dict)
    area_default:  int            = 0
    keywords: list[str] = Field(default_factory=list)
    loader:   Optional[PluginSpecModel] = None
    pre_cbs:  list[PluginSpecModel]     = Field(default_factory=list)
    post_cbs: list[PluginSpecModel]     = Field(default_factory=list)

    @classmethod
    def from_yaml(cls, path: str | Path) -> "HandlerConfig":
        "Load and validate a handler YAML config with Gate 1 static quarantine."
        raw = yaml.safe_load(Path(path).read_text(encoding="utf-8"))
        try:
            contract = _RawHandlerContract.model_validate(raw)
            return cls(
                module_name      = contract.handler.module_name,
                title            = contract.handler.title,
                description      = contract.handler.description.strip(),
                url              = contract.data_source.url,
                fname_out        = contract.data_source.fname_out,
                zenodo_id        = contract.data_source.zenodo_id,
                fmt              = contract.data_source.format,
                rename           = contract.rename_cols.mapping,
                string_cast      = contract.rename_cols.string_cast,
                columns          = contract.columns,
                normalize_case   = contract.normalize_case,
                col_date         = contract.parse_datetime.col_date,
                col_time         = contract.parse_datetime.col_time,
                dt_format        = contract.parse_datetime.format,
                time_format      = contract.time_format,
                meta_cols        = contract.melt.meta_cols,
                melt_spec        = contract.melt.spec,
                unit_conversions = contract.unit_conversions,
                nuclide_lut      = contract.nomenclatures.nuclide_lut,
                unit_lut         = contract.nomenclatures.unit_lut,
                lab_lut          = contract.nomenclatures.lab_lut,
                lab_constants    = contract.nomenclatures.lab_constants,
                area_default     = contract.nomenclatures.area_default,
                keywords         = contract.output.keywords,
                loader           = contract.loader,
                pre_cbs          = contract.pre_cbs,
                post_cbs         = contract.post_cbs,
            )
        except ValidationError as err:
            msg = _yaml_step1_message(err)
            print(msg)
            raise ValueError(msg) from err


In [ ]:
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
print(f"module_name:       {cfg.module_name}")
print(f"url:               {cfg.url[:55]}...")
print(f"rename keys:       {list(cfg.rename)[:3]}")
print(f"melt_spec entries: {len(cfg.melt_spec)}  (first: {cfg.melt_spec[0].val!r})")
print(f"unit_conversions:  {len(cfg.unit_conversions)}  (factor: {cfg.unit_conversions[0].factor:.3e})")
print(f"nuclide_lut:       {cfg.nuclide_lut}")
print(f"col_date / fmt:    {cfg.col_date!r} / {cfg.dt_format!r}")
print("HandlerConfig.from_yaml ✓")

In [ ]:
# PluginSpec: pre_cbs/post_cbs default to empty lists
cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
assert cfg.pre_cbs  == [], f"Expected [], got {cfg.pre_cbs}"
assert cfg.post_cbs == [], f"Expected [], got {cfg.post_cbs}"

# Legacy path: still works unchanged
spec_path = PluginSpec(path="marisco.callbacks.shared.SoftShiftLonCB", args={"shift": 180.0})
assert spec_path.path == "marisco.callbacks.shared.SoftShiftLonCB"
assert spec_path.args == {"shift": 180.0}

# New name: shorthand resolution
spec_name = PluginSpec(name="SoftRegexTransformCB")
assert spec_name.name == "SoftRegexTransformCB"
assert spec_name.path is None

# New file+class: local dynamic load
spec_file = PluginSpec(**{"file": "local_cbs.py", "class": "MyLocalCB"})
assert spec_file.file  == "local_cbs.py"
assert spec_file.class_ == "MyLocalCB"

# Validation guard: bare PluginSpec with no strategy raises
from pydantic import ValidationError
try:
    PluginSpec(args={"x": 1})
    assert False, "Should have raised"
except ValidationError:
    pass

print("PluginSpec ✓ — path/name/file+class all validate; bare spec raises ValidationError")

## load_data

Lazy intake — fetches the provider CSV/TSV and wraps it in the `{grp: DataFrame}` contract.

In [ ]:
#| export
def load_data(cfg: HandlerConfig, grp: str = "SEAWATER") -> dict[str, pd.DataFrame]:
    "Fetch raw CSV/TSV from cfg.url and return {grp: DataFrame}."
    r = requests.get(cfg.url, timeout=60)
    r.raise_for_status()
    sep = "	" if cfg.fmt == "tsv" else ","
    return {grp: pd.read_csv(io.BytesIO(r.content), sep=sep)}

## gap_check

Fail-Fast sensor: raises `ValueError` with scaffold CBs when required MARIS columns will be absent.

In [ ]:
#| export
_MARIS_REQUIRED = frozenset({"LAT", "LON", "TIME", "NUCLIDE", "VALUE", "UNC", "UNIT"})
_MELT_PROVIDES  = frozenset({"NUCLIDE", "VALUE", "UNC", "UNIT"})

def _gap_skeleton(gaps: set[str]) -> str:
    return "\n\n".join(
        f"class Fill{g}CB(PerGroupCB):\n"
        f"    \"TODO: provide {g} — add to columns/rename_cols or as a standalone CB.\"\n"
        f"    grps = [\'SEAWATER\']\n"
        f"    def each_grp(self, grp, df, state: PipelineState): df[\'{g}\'] = None  # FIXME"
        for g in sorted(gaps)
    )

def _recipe_paths_for(gaps: set[str]) -> list[str]:
    return [path for keys, path in _RECIPE_HINTS.items() if gaps & keys]

def _gap_gate_message(cfg: HandlerConfig, gaps: set[str]) -> str:
    parts = [
        f"⚠  GAP in {cfg.title!r} — missing MARIS columns: {sorted(gaps)}",
        "Easy Path (Check columns mapping): First, verify if simply adding raw CSV column mappings to 'columns:' resolves this missing column.",
    ]
    recipe_paths = _recipe_paths_for(gaps)
    if recipe_paths:
        parts.append(
            "Medium Path (Apply standard recipes): If dealing with combined coordinates or value-unit formats, copy-paste standard recipes from config/recipes/. Available recipes: " + ", ".join(recipe_paths)
        )
    parts.extend([
        "Hard Path (Custom Skeleton): If your dataset has highly specific anomalies (for example, conditional sign inversion), implement a local custom callback using the skeleton below.",
        _gap_skeleton(gaps),
    ])
    return "\n\n".join(parts)

def gap_check(cfg: HandlerConfig) -> None:
    "Fail-Fast Gate 2: missing MARIS columns get progressive remediation guidance."
    if cfg.pre_cbs: return
    all_rename = {**cfg.columns, **cfg.rename}
    available = (set(all_rename.values())
                 | (_MELT_PROVIDES if cfg.melt_spec else set())
                 | ({"TIME"} if cfg.col_date else set()))
    gaps = _MARIS_REQUIRED - available
    if not gaps:
        return
    msg = _gap_gate_message(cfg, gaps)
    print(f"\n{msg}\n")
    raise ValueError(f"YAML spec missing mappings for: {sorted(gaps)}")


In [ ]:
# Valid config: all 7 MARIS columns covered by rename + melt + parse_datetime
cfg_ok = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg_ok)
print("gap_check(fram_strait) → passed ✓")

In [ ]:
# Minimal config with no columns at all: fires with skeleton CBs + ValueError
cfg_bad = HandlerConfig(
    module_name="test.handler",
    title="Minimal Test",
    url="http://example.com/data.csv",
    fname_out="test.nc",
)
try:
    gap_check(cfg_bad)
except ValueError as e:
    print(f"
ValueError raised ✓: {e}")